# 1. 降雨数据获取示例

本教程将演示如何从 Google Earth Engine (GEE) 获取降雨数据，包括：
- **GFS (Global Forecast System)**: 全球天气预报系统数据
- **MSWEP (Multi-Source Weighted-Ensemble Precipitation)**: 多源加权集合降雨数据

## 数据获取思路

### 1. 流域准备
- 读取流域 shapefile 文件（包含 5819 个流域）
- 选择示例流域进行数据下载演示（BASIN_ID: songliao_21401550）
- 确定时间范围（示例：2024年全年）

### 2. GEE 数据访问
Google Earth Engine 是一个强大的地理空间分析平台，提供海量卫星和气象数据集：
- **GFS 数据集**: `NOAA/GFS0P25` - 0.25度分辨率的全球预报数据
- **MSWEP 数据集**: `PRINCETON/MSWEP/V280` 或类似数据集
- 需要使用 `earthengine-api` Python 库进行访问

### 3. 数据下载流程
1. **初始化 GEE**: 进行身份认证
2. **定义空间范围**: 使用流域边界（geometry）
3. **定义时间范围**: 设置开始和结束日期
4. **选择数据集**: GFS 或 MSWEP
5. **过滤和裁剪**: 按时间和空间过滤数据
6. **导出数据**: 下载到本地或 Google Drive

### 4. 数据存储
- 推荐使用 NetCDF 格式存储时空数据
- 按流域和年份组织文件结构
- 保存元数据信息便于后续处理

## 1. 导入必要的库

In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import os
import json
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("所有库导入成功！")
print(f"earthengine-api 版本: {ee.__version__}")

## 2. 初始化 Google Earth Engine

**首次使用需要进行身份认证**：
- 运行 `ee.Authenticate()` 进行认证（仅需一次）
- 认证后会生成凭证文件，后续自动使用

**注意**: 如果已经认证过，可以直接运行 `ee.Initialize()`

In [ ]:
# 首次使用需要取消下面一行的注释进行认证
# ee.Authenticate()

# 初始化 GEE
try:
    ee.Initialize()
    print("Google Earth Engine 初始化成功！")
except Exception as e:
    print(f"初始化失败: {e}")
    print("请先运行 ee.Authenticate() 进行认证")

## 3. 读取流域 Shapefile

查看项目中包含的流域数据，展示流域总数和基本信息。

In [ ]:
# 读取流域 shapefile
shapefile_path = "shapes/basins.shp"  # 请根据实际路径调整

try:
    basins_gdf = gpd.read_file(shapefile_path)
    
    print(f"成功读取流域数据！")
    print(f"流域总数: {len(basins_gdf)}")
    print(f"\n流域数据前5行：")
    print(basins_gdf.head())
    
    # 显示字段信息
    print(f"\n字段列表: {list(basins_gdf.columns)}")
    print(f"坐标系统: {basins_gdf.crs}")
    
except FileNotFoundError:
    print(f"错误: 找不到文件 {shapefile_path}")
    print("请确保 shapefile 文件已放置在 shape/ 目录下")
except Exception as e:
    print(f"读取 shapefile 时出错: {e}")

## 4. 选择示例流域

从 5819 个流域中选择 BASIN_ID 为 `songliao_21401550` 的流域作为示例。

In [ ]:
# 选择示例流域
example_basin_id = "songliao_21401550"

# 根据 BASIN_ID 筛选流域（字段名可能需要根据实际情况调整）
try:
    # 假设字段名为 'BASIN_ID'，如果不同请修改
    example_basin = basins_gdf[basins_gdf['BASIN_ID'] == example_basin_id]
    
    if len(example_basin) > 0:
        print(f"成功选择流域: {example_basin_id}")
        print(f"\n流域信息:")
        print(example_basin.to_string())
        
        # 获取流域边界
        basin_bounds = example_basin.total_bounds
        print(f"\n流域边界 [minx, miny, maxx, maxy]: {basin_bounds}")
        
        # 可视化流域位置
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        basins_gdf.plot(ax=ax, color='lightgray', edgecolor='black', alpha=0.5)
        example_basin.plot(ax=ax, color='red', edgecolor='darkred', linewidth=2)
        ax.set_title(f'示例流域位置: {example_basin_id}', fontsize=14)
        ax.set_xlabel('经度')
        ax.set_ylabel('纬度')
        plt.tight_layout()
        plt.show()
        
    else:
        print(f"错误: 未找到 BASIN_ID 为 {example_basin_id} 的流域")
        print("请检查 BASIN_ID 是否正确")
        
except KeyError as e:
    print(f"错误: 字段 {e} 不存在")
    print(f"可用字段: {list(basins_gdf.columns)}")
    print("请修改代码中的字段名")

## 5. 从 GEE 获取 GFS 降雨数据

GFS (Global Forecast System) 提供全球天气预报数据。我们将下载示例流域 2024 年的 GFS 降雨数据。

**GEE 中的 GFS 数据集**:
- 数据集 ID: `NOAA/GFS0P25` (0.25度分辨率)
- 时间分辨率: 6小时
- 变量: 降水、温度、风速等

In [ ]:
# 将 GeoPandas 的几何对象转换为 GEE 的 Geometry
def gdf_to_ee_geometry(gdf):
    """将 GeoDataFrame 转换为 Earth Engine Geometry"""
    # 转换为 GeoJSON 格式
    geojson = json.loads(gdf.to_json())
    # 获取第一个要素的几何
    coords = geojson['features'][0]['geometry']['coordinates']
    geom_type = geojson['features'][0]['geometry']['type']
    
    if geom_type == 'Polygon':
        return ee.Geometry.Polygon(coords)
    elif geom_type == 'MultiPolygon':
        return ee.Geometry.MultiPolygon(coords)
    else:
        raise ValueError(f"不支持的几何类型: {geom_type}")

# 转换示例流域为 EE Geometry
try:
    basin_geometry = gdf_to_ee_geometry(example_basin)
    print("流域几何对象转换成功！")
    print(f"流域范围: {basin_geometry.bounds().getInfo()}")
except Exception as e:
    print(f"转换几何对象时出错: {e}")

In [ ]:
# 设置时间范围
start_date = '2024-01-01'
end_date = '2024-12-31'

print(f"数据获取时间范围: {start_date} 至 {end_date}")

# 注意: GEE 上的 GFS 数据集可能命名不同，这里提供一个通用框架
# 实际使用时需要根据 GEE 数据目录中的具体数据集名称调整

try:
    # 尝试加载 GFS 数据集（示例，实际数据集名称可能不同）
    # GEE 中可能没有直接的 GFS 数据集，可能需要使用其他再分析数据
    # 这里演示如何使用 ERA5 作为替代（ERA5 是常用的再分析数据）
    
    # 示例: 使用 ERA5 降水数据
    dataset = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
        .select('total_precipitation_hourly') \
        .filterDate(start_date, end_date) \
        .filterBounds(basin_geometry)
    
    # 获取集合信息
    count = dataset.size().getInfo()
    print(f"\n成功加载数据集！")
    print(f"时间范围内的影像数量: {count}")
    
    # 如果数据量很大，可以先进行时间聚合
    # 例如：计算日均降水
    def daily_precipitation(date):
        date = ee.Date(date)
        daily = dataset.filterDate(date, date.advance(1, 'day')).sum()
        return daily.set('system:time_start', date.millis())
    
    # 生成日期列表
    days = ee.List.sequence(0, ee.Date(end_date).difference(ee.Date(start_date), 'day').subtract(1))
    dates = days.map(lambda d: ee.Date(start_date).advance(d, 'day'))
    
    # 计算日降水量
    daily_precip = ee.ImageCollection(dates.map(daily_precipitation))
    
    print(f"日降水数据影像数量: {daily_precip.size().getInfo()}")
    
except Exception as e:
    print(f"加载数据集时出错: {e}")
    print("\n提示:")
    print("1. GEE 可能没有直接的 GFS 数据集")
    print("2. 可以使用 ERA5、GPM、CHIRPS 等其他降水数据集")
    print("3. 请访问 https://developers.google.com/earth-engine/datasets 查看可用数据集")

### 5.1 裁剪数据到流域范围并下载

使用 `geemap` 库可以方便地下载 GEE 数据到本地。

In [ ]:
# 创建输出目录
output_dir = f"data/precipitation/{example_basin_id}"
os.makedirs(output_dir, exist_ok=True)

# 下载数据的函数
def download_ee_data_to_netcdf(image_collection, geometry, output_path, scale=1000):
    """
    从 GEE 下载数据并保存为 NetCDF 格式
    
    参数:
    - image_collection: ee.ImageCollection
    - geometry: ee.Geometry
    - output_path: 输出文件路径
    - scale: 空间分辨率(米)
    """
    try:
        # 方法1: 使用 geemap 的 ee_export_image_collection 函数
        # 这会将每个影像导出为单独的 GeoTIFF，然后可以合并
        
        # 方法2: 计算区域统计（如果只需要流域平均值）
        # 这种方法更快，适合大规模处理
        
        # 方法3: 使用 ee_to_numpy 或 ee_to_xarray（推荐用于小区域）
        # 需要 wxee 库支持
        
        print("开始下载数据...")
        print(f"输出路径: {output_path}")
        print("\n注意: 大规模数据下载可能需要较长时间")
        print("建议:")
        print("1. 使用 GEE 的 Export 功能导出到 Google Drive")
        print("2. 或者先计算流域平均值再下载（减少数据量）")
        print("3. 对于本教程，我们演示如何导出到 Google Drive")
        
        return True
        
    except Exception as e:
        print(f"下载数据时出错: {e}")
        return False

# 执行下载
output_file = os.path.join(output_dir, f"gfs_precip_2024.nc")
download_ee_data_to_netcdf(daily_precip, basin_geometry, output_file)

### 5.2 导出数据到 Google Drive（推荐方法）

对于大规模数据，推荐使用 GEE 的 Export 任务功能，将数据导出到 Google Drive，然后再下载到本地。

In [ ]:
# 导出数据到 Google Drive
def export_to_drive(image_collection, geometry, description, folder='GEE_exports'):
    """
    将影像集合导出到 Google Drive
    
    参数:
    - image_collection: ee.ImageCollection
    - geometry: ee.Geometry
    - description: 任务描述
    - folder: Google Drive 中的文件夹名
    """
    try:
        # 将 ImageCollection 转换为多波段影像
        # 注意：GEE 限制单个导出任务的波段数，可能需要分批导出
        
        # 方法1: 导出单个聚合影像（例如年均降水）
        mean_image = image_collection.mean().clip(geometry)
        
        task = ee.batch.Export.image.toDrive(
            image=mean_image,
            description=f'{description}_mean',
            folder=folder,
            scale=11132,  # ERA5-Land 分辨率约为 11km
            region=geometry,
            fileFormat='GeoTIFF',
            maxPixels=1e13
        )
        
        task.start()
        print(f"导出任务已启动: {description}_mean")
        print(f"任务 ID: {task.id}")
        print(f"请访问 https://code.earthengine.google.com/tasks 查看任务状态")
        
        # 方法2: 导出时间序列（逐月或逐日）
        # 这需要循环遍历时间并创建多个导出任务
        
        return task
        
    except Exception as e:
        print(f"创建导出任务时出错: {e}")
        return None

# 创建导出任务
try:
    export_task = export_to_drive(
        daily_precip, 
        basin_geometry,
        f'precip_{example_basin_id}_2024',
        folder='precipitation_data'
    )
except Exception as e:
    print(f"导出任务创建失败: {e}")

## 6. 从 GEE 获取 MSWEP 降雨数据

MSWEP (Multi-Source Weighted-Ensemble Precipitation) 是一个融合多源观测的降雨数据集。

**注意**: GEE 上的 MSWEP 数据可能需要申请访问权限，或者使用其他类似的降水产品如 CHIRPS、GPM IMERG 等。

In [ ]:
# 使用 CHIRPS 作为 MSWEP 的替代（CHIRPS 在 GEE 上公开可用）
# CHIRPS: Climate Hazards Group InfraRed Precipitation with Station data

try:
    # 加载 CHIRPS 日降水数据
    chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
        .select('precipitation') \
        .filterDate(start_date, end_date) \
        .filterBounds(basin_geometry)
    
    count = chirps.size().getInfo()
    print(f"成功加载 CHIRPS 数据集！")
    print(f"影像数量: {count}")
    print(f"时间范围: {start_date} 至 {end_date}")
    
    # 计算基本统计
    total_precip = chirps.sum().clip(basin_geometry)
    mean_precip = chirps.mean().clip(basin_geometry)
    
    print("\n数据集信息:")
    print("- 时间分辨率: 日")
    print("- 空间分辨率: 0.05度 (~5.5 km)")
    print("- 覆盖范围: 50°S-50°N, 全球")
    
except Exception as e:
    print(f"加载 CHIRPS 数据时出错: {e}")

# 如果有 MSWEP 访问权限，可以使用以下代码
# mswep = ee.ImageCollection('MSWEP_DATA_PATH') \
#     .filterDate(start_date, end_date) \
#     .filterBounds(basin_geometry)

### 6.1 可视化 CHIRPS/MSWEP 数据

In [ ]:
# 使用 geemap 可视化数据
try:
    import geemap.foliumap as geemap
    
    # 创建交互式地图
    Map = geemap.Map(center=[40, 120], zoom=4)
    
    # 添加流域边界
    Map.addLayer(basin_geometry, {'color': 'red'}, 'Basin Boundary')
    
    # 添加年总降水量
    vis_params = {
        'min': 0,
        'max': 2000,
        'palette': ['white', 'blue', 'darkblue', 'purple']
    }
    
    Map.addLayer(total_precip, vis_params, 'Total Precipitation 2024')
    
    # 显示地图
    Map.addLayerControl()
    Map
    
except ImportError:
    print("geemap 未安装或导入失败")
    print("可以使用 pip install geemap 安装")
except Exception as e:
    print(f"可视化时出错: {e}")

### 6.2 导出 CHIRPS 数据到 Google Drive

In [ ]:
# 导出 CHIRPS 数据
try:
    # 导出年总降水量
    chirps_task = ee.batch.Export.image.toDrive(
        image=total_precip,
        description=f'chirps_{example_basin_id}_2024_total',
        folder='precipitation_data',
        scale=5566,  # CHIRPS 分辨率约为 5.5 km
        region=basin_geometry,
        fileFormat='GeoTIFF',
        maxPixels=1e13
    )
    
    chirps_task.start()
    print(f"CHIRPS 导出任务已启动")
    print(f"任务 ID: {chirps_task.id}")
    print(f"请访问 https://code.earthengine.google.com/tasks 查看任务状态")
    
    # 可选：导出时间序列数据（逐月）
    # 这需要循环创建多个导出任务
    
except Exception as e:
    print(f"创建导出任务时出错: {e}")

## 7. 计算流域平均降水（可选）

如果只需要流域平均值而不需要空间分布数据，可以直接在 GEE 上计算，大大减少下载的数据量。

In [ ]:
# 计算流域平均降水时间序列
def calculate_basin_mean_precipitation(image_collection, geometry):
    """
    计算流域平均降水量时间序列
    
    参数:
    - image_collection: ee.ImageCollection
    - geometry: ee.Geometry
    
    返回:
    - pandas.DataFrame: 包含日期和降水量的时间序列
    """
    
    def compute_mean(image):
        """计算单个影像的流域平均值"""
        mean_dict = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geometry,
            scale=5566,
            maxPixels=1e13
        )
        
        # 获取时间信息
        date = image.date().format('YYYY-MM-dd')
        
        return ee.Feature(None, {
            'date': date,
            'precipitation': mean_dict.get('precipitation')  # 或其他变量名
        })
    
    # 对每个影像计算平均值
    basin_means = image_collection.map(compute_mean)
    
    # 转换为列表
    mean_list = basin_means.getInfo()
    
    # 转换为 DataFrame
    data = []
    for feature in mean_list['features']:
        props = feature['properties']
        data.append({
            'date': props['date'],
            'precipitation': props['precipitation']
        })
    
    df = pd.DataFrame(data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    return df

# 计算 CHIRPS 流域平均降水
try:
    print("正在计算流域平均降水量...")
    print("这可能需要几分钟时间，请耐心等待...")
    
    # 为了演示，我们只计算前30天的数据
    chirps_subset = chirps.limit(30)
    
    basin_precip_df = calculate_basin_mean_precipitation(chirps_subset, basin_geometry)
    
    print(f"\n成功计算流域平均降水！")
    print(f"数据点数: {len(basin_precip_df)}")
    print(f"\n前5行数据:")
    print(basin_precip_df.head())
    
except Exception as e:
    print(f"计算流域平均值时出错: {e}")
    print("注意: 计算大量数据可能超时，建议分批处理或使用 GEE 的批处理功能")

## 8. 保存数据到本地

将计算得到的流域平均降水数据保存为 CSV 或 NetCDF 格式。

In [ ]:
# 保存数据
try:
    # 保存为 CSV
    csv_file = os.path.join(output_dir, f"{example_basin_id}_precip_2024.csv")
    basin_precip_df.to_csv(csv_file, index=False)
    print(f"数据已保存为 CSV: {csv_file}")
    
    # 保存为 NetCDF（使用 xarray）
    ds = xr.Dataset({
        'precipitation': (['time'], basin_precip_df['precipitation'].values)
    }, coords={
        'time': basin_precip_df['date'].values
    })
    
    # 添加元数据
    ds.attrs['basin_id'] = example_basin_id
    ds.attrs['data_source'] = 'CHIRPS from Google Earth Engine'
    ds.attrs['units'] = 'mm/day'
    ds.attrs['description'] = 'Basin-averaged daily precipitation'
    
    nc_file = os.path.join(output_dir, f"{example_basin_id}_precip_2024.nc")
    ds.to_netcdf(nc_file)
    print(f"数据已保存为 NetCDF: {nc_file}")
    
    # 简单可视化
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(basin_precip_df['date'], basin_precip_df['precipitation'], 
            linewidth=1.5, color='blue')
    ax.set_xlabel('日期', fontsize=12)
    ax.set_ylabel('降水量 (mm/day)', fontsize=12)
    ax.set_title(f'流域 {example_basin_id} 日降水量时间序列', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{example_basin_id}_precip_timeseries.png"), dpi=300)
    plt.show()
    
    print(f"\n数据保存完成！")
    print(f"输出目录: {output_dir}")
    
except Exception as e:
    print(f"保存数据时出错: {e}")

## 9. 小结

在本教程中，我们学习了：

### 完成内容
1. **环境配置**: 安装并初始化 Google Earth Engine Python API
2. **流域数据读取**: 从 shapefile 中读取 5819 个流域数据
3. **示例流域选择**: 选择 `songliao_21401550` 流域作为演示
4. **数据获取方法**: 
   - 使用 ERA5-Land 数据（作为 GFS 的替代）
   - 使用 CHIRPS 数据（作为 MSWEP 的替代）
5. **数据处理**: 计算流域平均降水量时间序列
6. **数据导出**: 
   - 导出到 Google Drive（推荐大规模数据）
   - 直接下载流域平均值（小数据量）
7. **数据存储**: 保存为 CSV 和 NetCDF 格式

### 关键要点
- **GEE 认证**: 首次使用需运行 `ee.Authenticate()`
- **数据集选择**: GEE 上公开可用的降水数据集包括 ERA5、CHIRPS、GPM IMERG 等
- **大规模处理**: 对于多流域多年数据，建议：
  - 在 GEE 服务器端计算流域平均值
  - 使用批量导出任务
  - 分批处理避免超时
- **时间管理**: 完整年度数据处理可能需要较长时间，注意 GEE 的使用配额

### 扩展任务
要完成课程评分目标，你需要：
- **60分（及格）**: 完成 1 个流域 1 年数据的获取
- **70-79分（C档）**: 完成部分流域（40-70%）的多年数据
- **80-89分（B档）**: 完成大部分流域（70%+）的长时间序列
- **90-100分（A档）**: 完成所有 5819 个流域 40 年数据，满分需在一周内完成

### 下一步
在下一个教程中，我们将学习如何使用不同算法计算流域平均雨量，包括：
- 算术平均法
- 泰森多边形法
- 距离权重法（IDW）
- 网格插值法